# 01 — Data Exploration
**Project:** Linear Algebra-Based Discovery of Disease-Linked Gene Expression Patterns

This notebook loads a GEO dataset, inspects its structure, and produces initial QC plots.

In [ ]:
# ── 0. Setup ──────────────────────────────────────────────────────────────
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

pip('GEOparse')
pip('umap-learn')


In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import GEOparse

# Mount Google Drive (uncomment if needed)
# from google.colab import drive
# drive.mount('/content/drive')

# Project root – adjust if you cloned the repo to Drive
PROJECT_ROOT = '/content/gene-expression-la-analysis'
os.makedirs(f'{PROJECT_ROOT}/data/raw',       exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data/processed', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data/results',   exist_ok=True)

# Add src/ to path
sys.path.insert(0, f'{PROJECT_ROOT}/src')
from preprocessing import qc_report

sns.set_theme(style='whitegrid', font_scale=1.1)
print('Setup complete.')


## 1. Download GEO Dataset
We use **GSE2034** (breast cancer microarray, 286 samples) as a well-known benchmark.
Change `GEO_ID` to any GSE accession you prefer.

In [ ]:
GEO_ID   = 'GSE2034'          # <-- change to your dataset
RAW_DIR  = f'{PROJECT_ROOT}/data/raw'

gse = GEOparse.get_GEO(geo=GEO_ID, destdir=RAW_DIR, silent=True)
print(f'Loaded {GEO_ID}: {len(gse.gsms)} samples, {len(gse.gpls)} platform(s)')


In [ ]:
# ── Build expression matrix (probes × samples) ────────────────────────────
pivot = gse.pivot_samples('VALUE')   # returns probes × samples DataFrame
print('Expression matrix shape:', pivot.shape)
pivot.head(3)


In [ ]:
# ── Extract phenotype / metadata ───────────────────────────────────────────
meta_rows = []
for name, gsm in gse.gsms.items():
    row = {'sample': name}
    row.update(gsm.metadata)
    meta_rows.append(row)

meta = pd.DataFrame(meta_rows).set_index('sample')
# Flatten list-valued columns
for col in meta.columns:
    meta[col] = meta[col].apply(lambda x: x[0] if isinstance(x, list) else x)

print('Metadata shape:', meta.shape)
meta[['title', 'source_name_ch1']].head(5)


## 2. QC Report

In [ ]:
expr = pivot.apply(pd.to_numeric, errors='coerce')
report = qc_report(expr)


In [ ]:
# ── Distribution of per-sample means ──────────────────────────────────────
sample_means = expr.mean(axis=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(sample_means, bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Sample Means')
axes[0].set_xlabel('Mean Expression')

# Per-gene variance distribution
gene_vars = expr.var(axis=1)
axes[1].hist(np.log10(gene_vars.clip(lower=1e-6)), bins=60,
             color='teal', edgecolor='white')
axes[1].set_title('log10(Gene Variance)')
axes[1].set_xlabel('log10 Variance')

# Missing values per sample
missing = expr.isnull().sum(axis=0)
axes[2].bar(range(len(missing)), missing.values, color='coral')
axes[2].set_title('Missing Values per Sample')
axes[2].set_xlabel('Sample index')

plt.suptitle('Initial QC Overview', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/results/01_qc_overview.png',
            dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Box-plot of first 40 samples (spot-check normalisation) ───────────────
fig, ax = plt.subplots(figsize=(14, 5))
sample_data = [expr.iloc[:, i].dropna().values for i in range(min(40, expr.shape[1]))]
ax.boxplot(sample_data, patch_artist=True,
           boxprops=dict(facecolor='lightsteelblue', alpha=0.7),
           medianprops=dict(color='crimson', lw=1.5),
           flierprops=dict(marker='.', ms=2, alpha=0.3))
ax.set_title('Per-sample Expression Distributions (first 40 samples)')
ax.set_xlabel('Sample')
ax.set_ylabel('Expression value')
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/results/01_sample_boxplot.png',
            dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Sample–sample correlation heatmap (subset) ────────────────────────────
subset = expr.iloc[:, :60].dropna()
corr   = subset.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0, ax=ax,
            cbar_kws={'shrink': 0.7})
ax.set_title('Sample–Sample Pearson Correlation (first 60 samples)')
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/results/01_sample_corr_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Save raw matrix for downstream notebooks ───────────────────────────────
out_path = f'{PROJECT_ROOT}/data/raw/{GEO_ID}_expression_raw.csv'
expr.to_csv(out_path)
meta.to_csv(f'{PROJECT_ROOT}/data/raw/{GEO_ID}_metadata.csv')
print(f'Saved raw expression matrix → {out_path}')


## Summary

| Item | Value |
|------|-------|
| Dataset | GSE2034 |
| Probes | see report |
| Samples | see report |
| Missing % | see report |

**Next:** `02_preprocessing.ipynb` — normalisation and QC filtering.